# Cap 07 — Paralelización y Fan-out

Ejecutamos múltiples análisis en paralelo para reducir la latencia:
- Ramas paralelas estáticas (fan-out fijo)
- `Send()`: despacho dinámico de workers
- Benchmark: paralelo vs secuencial

**Dominio**: Álbumes de Miles Davis (análisis multi-dimensional)  
**Flujo estático**: `START → [armonia|ritmo|influencia] → sintetizador → END`  
**Flujo dinámico**: `START → selector → Send(workers) → aggregator → END`

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import TypedDict, List, Annotated
from operator import add
from dotenv import load_dotenv

load_dotenv()

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
from rich import print as rprint

DATA_PATH = Path("../00_datos/davis_albums.json")
davis_albums = json.loads(DATA_PATH.read_text(encoding="utf-8"))

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), temperature=0)
print("Dependencias cargadas ✓")

## 1. El problema secuencial

Si necesitamos 3 análisis independientes, el enfoque secuencial suma todas las latencias.

In [ ]:
kind_of_blue = next(a for a in davis_albums if "Kind of Blue" in a["titulo"])

# Simulamos análisis secuencial
def analizar_aspecto(album: dict, aspecto: str) -> str:
    response = llm.invoke([
        SystemMessage(content=f"Eres un musicólogo analizando el aspecto '{aspecto}' de un álbum de jazz."),
        HumanMessage(content=f"Analiza el aspecto '{aspecto}' de:\n{json.dumps(album, ensure_ascii=False)}")
    ])
    return response.content

print("Midiendo tiempo secuencial...")
t0 = time.time()
r_armonia = analizar_aspecto(kind_of_blue, "harmonía modal")
r_ritmo = analizar_aspecto(kind_of_blue, "estructura rítmica")
r_influencia = analizar_aspecto(kind_of_blue, "influencias e influenciados")
t_secuencial = time.time() - t0

print(f"Tiempo secuencial: {t_secuencial:.2f}s (3 llamadas en serie)")

## 2. Ramas Paralelas Estáticas

LangGraph ejecuta nodos sin dependencias entre sí en paralelo automáticamente.

In [ ]:
class DavisAnalysisState(TypedDict):
    album_data: dict
    query: str
    analysis_armonia: str
    analysis_ritmo: str
    analysis_influencia: str
    synthesis: str

def node_selector(state: DavisAnalysisState) -> dict:
    """Selecciona el álbum a analizar basado en la query."""
    query = state["query"].lower()
    album = next(
        (a for a in davis_albums if any(w in a["titulo"].lower() for w in query.split())),
        davis_albums[0]
    )
    return {"album_data": album}

def make_analysis_node(aspect: str):
    def node(state: DavisAnalysisState) -> dict:
        album = state["album_data"]
        if not album:
            return {}
        response = llm.invoke([
            SystemMessage(content=f"Eres musicólogo. Analiza el aspecto '{aspect}' en 2-3 oraciones."),
            HumanMessage(content=f"Álbum: {json.dumps({'titulo': album['titulo'], 'descripcion': album.get('descripcion', '')}, ensure_ascii=False)}")
        ])
        key = f"analysis_{aspect.replace(' ', '_').replace('é', 'e').replace('ó', 'o')}"
        return {key: response.content}
    return node

node_armonia = make_analysis_node("armonía")
node_ritmo = make_analysis_node("ritmo")
node_influencia = make_analysis_node("influencia")

def node_sintetizador(state: DavisAnalysisState) -> dict:
    """Combina los 3 análisis paralelos en un párrafo cohesivo usando el LLM."""
    album_titulo = state.get("album_data", {}).get("titulo", "el álbum")
    analisis_combinado = f"""
Armonía: {state.get("analysis_armonia", "N/A")}

Ritmo: {state.get("analysis_ritmo", "N/A")}

Influencia: {state.get("analysis_influencia", "N/A")}
"""
    response = llm.invoke([
        SystemMessage(content=(
            "Eres un musicólogo que integra análisis especializados en una síntesis cohesiva. "
            "Escribe un único párrafo fluido que conecte los aspectos de armonía, ritmo e influencia "
            "sin usar bullet points ni secciones separadas. El párrafo debe mostrar cómo estos "
            "tres aspectos se interrelacionan para definir la obra."
        )),
        HumanMessage(content=f"Sintetiza estos análisis sobre '{album_titulo}':\n{analisis_combinado}")
    ])
    return {"synthesis": response.content}

# Construir grafo con ramas paralelas
builder = StateGraph(DavisAnalysisState)
builder.add_node("selector", node_selector)
builder.add_node("armonia", node_armonia)
builder.add_node("ritmo", node_ritmo)
builder.add_node("influencia", node_influencia)
builder.add_node("sintetizador", node_sintetizador)

builder.add_edge(START, "selector")
# Fan-out: selector → [armonia, ritmo, influencia] en paralelo
builder.add_edge("selector", "armonia")
builder.add_edge("selector", "ritmo")
builder.add_edge("selector", "influencia")
# Fan-in: todos convergen en sintetizador
builder.add_edge("armonia", "sintetizador")
builder.add_edge("ritmo", "sintetizador")
builder.add_edge("influencia", "sintetizador")
builder.add_edge("sintetizador", END)

graph_paralelo = builder.compile()
print("Grafo con ramas paralelas compilado ✓")

In [ ]:
print("Midiendo tiempo paralelo...")
t0 = time.time()
result_paralelo = graph_paralelo.invoke({
    "query": "Kind of Blue",
    "album_data": {},
    "analysis_armonia": "", "analysis_ritmo": "", "analysis_influencia": "",
    "synthesis": ""
})
t_paralelo = time.time() - t0

print(f"Tiempo paralelo: {t_paralelo:.2f}s")
print(f"Speedup: {t_secuencial/t_paralelo:.1f}x más rápido")
print("\n" + result_paralelo["synthesis"])

## 3. `Send()`: Despacho Dinámico de Workers

`Send()` permite crear workers dinámicamente basado en los datos del estado.

In [ ]:
class AlbumWorkerState(TypedDict):
    """Estado para cada worker de álbum."""
    album: dict
    analysis: str

class AggregatorState(TypedDict):
    """Estado del aggregator."""
    albums_to_analyze: List[str]
    results: Annotated[List[str], add]  # add reducer: cada worker añade su resultado

def node_dispatcher(state: AggregatorState):
    """Despacha un worker por álbum usando Send()."""
    albums = [a for a in davis_albums if a["titulo"] in state["albums_to_analyze"]]
    return [
        Send("album_worker", {"album": album, "analysis": ""})
        for album in albums
    ]

def node_album_worker(state: AlbumWorkerState) -> dict:
    """Worker: analiza un único álbum."""
    album = state["album"]
    response = llm.invoke([
        SystemMessage(content="Resume la importancia histórica de este álbum de Miles Davis en 1 oración."),
        HumanMessage(content=f"{album['titulo']} ({album['año']}): {album.get('descripcion', '')[:200]}")
    ])
    result = f"• **{album['titulo']}** ({album['año']}): {response.content}"
    return {"results": [result]}

def node_aggregator(state: AggregatorState) -> dict:
    """Agrega los resultados de todos los workers."""
    return {"results": state["results"]}  # Ya agregados por el reducer

builder2 = StateGraph(AggregatorState)
builder2.add_node("album_worker", node_album_worker)
builder2.add_node("aggregator", node_aggregator)

builder2.add_conditional_edges(START, node_dispatcher, ["album_worker"])
builder2.add_edge("album_worker", "aggregator")
builder2.add_edge("aggregator", END)

graph_dinamico = builder2.compile()
print("Grafo dinámico con Send() compilado ✓")

In [ ]:
t0 = time.time()
result_dinamico = graph_dinamico.invoke({
    "albums_to_analyze": ["Kind of Blue", "Bitches Brew", "Birth of the Cool", "Sketches of Spain"],
    "results": []
})
t_dinamico = time.time() - t0

print(f"Tiempo con 4 álbumes en paralelo: {t_dinamico:.2f}s")
print("\n=== Análisis Paralelo de 4 Álbumes ===")
for r in result_dinamico["results"]:
    print(r)

## Resumen del Capítulo

| Concepto | Descripción |
|---------|-------------|
| Fan-out estático | `selector → [A, B, C]` ejecuta A, B, C en paralelo |
| Fan-in | `[A, B, C] → aggregator` espera a todos |
| `Send(node, state)` | Despacha dinámicamente un worker con estado propio |
| `Annotated[list, add]` | Reducer que acumula resultados de múltiples workers |
| `WorkerState` | TypedDict separado para el estado de cada worker |

**Próximo capítulo**: Sistema multi-agente con sub-grafos